In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing  import StandardScaler

In [0]:
df = pd.read_csv('/Workspace/Users/gusemenuk4@gmail.com/Tech_Challenge_Fase_3/data/features/features.csv')
df

In [0]:
df["atingiu_meta"] = (
    df["pc_indicador_alfabetizacao"] >= 0.80
).astype(int)

In [0]:
X = df[['vl_proficiencia_media','vl_proficiencia_mediana','qt_alunos_avaliados','nu_serie','qtd_fam_pobreza','qtd_fam_baixa_renda','qtd_fam_ate_meio_sm','qtd_fam_renda_zero','taxa_atualizacao_geral_pct','taxa_atualizacao_ate_meio_sm_pct','qtd_escolas','pct_urbana','pct_rural','ano_2024','ds_rede_municipal','ds_rede_privada','nome_regiao_nordeste','nome_regiao_norte','nome_regiao_sudeste','nome_regiao_sul','uf_freq']] 
y = df['atingiu_meta']

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print('Treino:',X_train.shape)
print('Teste:', X_test.shape)


In [0]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [0]:
log_model = LogisticRegression()

log_model.fit(X_train_scaled,y_train)

pred_log = log_model.predict(X_test_scaled)

In [0]:
y_proba = log_model.predict_proba(X_test_scaled)[:, 1]

print(y_proba[:10])

In [0]:
acc_log = accuracy_score(y_test, pred_log)

print('Acurácia:', acc_log)

In [0]:
print(df['atingiu_meta'].value_counts())

print("\nProporção:")
print(df['atingiu_meta'].value_counts(normalize=True))

In [0]:
cm = confusion_matrix(y_test, pred_log)

print(cm)

In [0]:
precision = precision_score(y_test, pred_log)

print("Precision:", precision)

In [0]:
recall = recall_score(y_test, pred_log)

print("Recall:", recall)

In [0]:
f1 = f1_score(y_test, pred_log)

print("F1-Score:", f1)

In [0]:
print(classification_report(y_test, pred_log))

In [0]:
auc = roc_auc_score(y_test, y_proba)

print("ROC-AUC:", auc)

In [0]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(8, 6))

plt.plot(fpr, tpr)

plt.plot([0, 1], [0, 1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC")

plt.show()

### Feature Importance

In [0]:
log_model.fit(X_train_scaled, y_train)

coeficientes = log_model.coef_[0]

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coeficiente': coeficientes
})

feature_importance

In [0]:
feature_importance['importance'] = (
    feature_importance['coeficiente'].abs()
)

In [0]:
feature_importance = feature_importance.sort_values(
    'importance',
    ascending=False
)

feature_importance

In [0]:
plt.figure(figsize=(10, 8))

plt.barh(
    feature_importance['feature'],
    feature_importance['coeficiente']
)

plt.xlabel('Coeficiente')
plt.ylabel('Feature')
plt.title('Importância das Features - Regressão Logística')

plt.axvline(0, linestyle='--')

plt.gca().invert_yaxis()

plt.show()

In [0]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coeficiente': log_model.coef_[0]
})

feature_importance['importance'] = (
    feature_importance['coeficiente'].abs()
)

feature_importance['odds_ratio'] = (
    np.exp(feature_importance['coeficiente'])
)

feature_importance = feature_importance.sort_values(
    'importance',
    ascending=False
)

feature_importance